## 1.边特征独热编码处理
- signature.csv
- port_service.csv
- encode_day_time.csv

## 1.1 处理signature.csv

In [ ]:
import pandas as pd
# 读入csv

file_name_list = [
    'monday-alert-label.csv',
   'tuesday-alert-label.csv',
   'wednesday-alert-label.csv',
   'thursday-alert-label.csv',
   'friday-enhance-portscan-alert-label.csv'
]
merge_df = pd.DataFrame()
for file_name in file_name_list:
    # 读取CSV文件
    file_path = f'../../alert_label_merge_csv/{file_name}'
    df = pd.read_csv(file_path)
    # 合并
    merge_df = pd.concat([merge_df, df], ignore_index=True)
merge_df['rule'].value_counts()   

1:40063:5     175370
119:201:1     107102
119:260:1      83658
1:22953:5      67766
1:27938:3      59699
               ...  
1:38381:4          1
1:38767:2          1
1:8414:16          1
1:19072:16         1
1:16231:22         1
Name: rule, Length: 177, dtype: int64

In [15]:
# 仅保留rule,calss,message三列
signature_df = merge_df[['rule','class','msg']].copy()
signature_df.reset_index(drop=True,inplace=True)
# 去重
signature_df.drop_duplicates(inplace=True)
signature_df.reset_index(drop=True,inplace=True)
# 产生ID列，从1开始
signature_df['ID'] = range(1, len(signature_df) + 1)

In [16]:
# 将sig_id列转换为字符串类型
signature_df['ID'] = signature_df['ID'].astype(str)

# 对id列进行one-hot编码
one_hot_encoded = pd.get_dummies(signature_df['ID'])
# 获取独热编码的列名并按照原始数据顺序重新排序
columns_sorted = sorted(one_hot_encoded.columns, key=lambda x: int(x.split('_')[-1]))
# 重新排序独热编码的列
one_hot_encoded_sorted = one_hot_encoded[columns_sorted]
# 将编码列合并为一列，形成列表
signature_df['one_hot'] = one_hot_encoded_sorted.apply(lambda row: row.values.tolist(), axis=1)
signature_df

,rule,class,msg,ID,one_hot
0,119:228:1,none,(http_inspect) server response before client r...,1,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1:57878:1,Attempted User Privilege Gain,PROTOCOL-DNS Microsoft Threat Management Gatew...,2,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,129:15:1,none,(stream_tcp) reset outside window,3,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,119:201:1,none,(http_inspect) not HTTP traffic or unrecoverab...,4,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,1:57756:2,A Network Trojan was detected,MALWARE-CNC DNS Fast Flux attempt,5,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...
172,116:434:1,none,(icmp4) ICMP ping Nmap,173,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
173,1:25314:8,Detection of a Denial of Service Attack,OS-LINUX Linux kernel IGMP queries denial of s...,174,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
174,1:3822:15,Generic Protocol Command Decode,SERVER-WEBAPP RealNetworks RealPlayer realtext...,175,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
175,1:16301:15,Attempted User Privilege Gain,BROWSER-IE Microsoft Internet Explorer HTML DO...,176,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [17]:
# 读出csv
signature_df.to_csv("encoded_signature.csv",index=False)

### 1.2 处理port_service.csv

In [2]:
import pandas as pd
# 读入csv
port_path = "../../"
port_df = pd.read_csv(port_path)
port_df.reset_index(inplace=True)
port_df.rename(columns={'index': 'id'}, inplace=True)

# 将id列转换为字符串类型
port_df['id'] = port_df['id'].astype(str)

# 对id列进行one-hot编码
one_hot_encoded = pd.get_dummies(port_df['id'])

# 获取独热编码的列名并按照原始数据顺序重新排序
columns_sorted = sorted(one_hot_encoded.columns, key=lambda x: int(x.split('_')[-1]))

# 重新排序独热编码的列
one_hot_encoded_sorted = one_hot_encoded[columns_sorted]

# 将编码列合并为一列，形成列表
port_df['one_hot'] = one_hot_encoded_sorted.apply(lambda row: row.values.tolist(), axis=1)

# 读出csv
port_df.to_csv("../../",index=False)

### 1.3 生成encode_day_time.csv
- 按照周一至周日+24小时进行分类
- 生成7+24 = 31维特征

In [69]:
week_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hours = list(range(24))

data = {
    'week_day': week_days * len(hours),
    'hour': hours * len(week_days)
}

df = pd.DataFrame(data)


# 添加新列来表示"week_day"的顺序
df['week_day'] = pd.Categorical(df['week_day'], categories=week_days, ordered=True)
day_time_df = df.sort_values(by=['week_day', 'hour'])


# 为day_time_df 添加新索引 
day_time_df = day_time_df.reset_index().drop('index',axis=1)
day_time_df = day_time_df.reset_index().rename(columns={'index': 'id'})

# 进行one-hot编码 
week_day_encoded = pd.get_dummies(day_time_df['week_day'])
hour_encoded = pd.get_dummies(day_time_df['hour'])

one_hot_df = pd.concat([week_day_encoded, hour_encoded], axis=1)

day_time_df['one_hot'] = one_hot_df.apply(lambda row: row.values.tolist(), axis=1)

# 读出csv
day_time_df.to_csv("encode_day_time.csv",index=False)

In [6]:
from datetime import datetime

# 创建一个datetime对象
dt = datetime(2023, 6, 6)

# 获取星期几
weekday = dt.weekday()

# 打印星期几
print(weekday)


1
